In [2]:
import pandas as pd

files = [
    "Monday-WorkingHours.pcap_ISCX.csv",
    "Tuesday-WorkingHours.pcap_ISCX.csv",
    "Friday-WorkingHours-Morning.pcap_ISCX.csv",
    "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv"
]

dfs = [pd.read_csv(f) for f in files]
combined_df = pd.concat(dfs, ignore_index=True)

combined_df.to_csv("cicids_combined.csv", index=False)

print("Combined dataset shape:", combined_df.shape)


Combined dataset shape: (1392605, 79)


In [3]:
import numpy as np
df = pd.read_csv("cicids_combined.csv")

# Normalize column names
df.columns = df.columns.str.strip().str.replace(" ", "_")

# Replace infinities
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop rows with NaNs
df.dropna(inplace=True)

print(df["Label"].value_counts())


Label
BENIGN         1247935
DDoS            128025
FTP-Patator       7935
SSH-Patator       5897
Bot               1956
Name: count, dtype: int64


In [6]:
df["Label"] = df["Label"].apply(
    lambda x: 0 if x == "BENIGN" else 1
)


Why?

Multi-class IDS is harder to defend

Binary detection is industry-default

Easier false-positive control

You can extend later.

In [8]:
features = [
    "Flow_Duration",
    "Total_Fwd_Packets",
    "Total_Backward_Packets",
    "Total_Length_of_Fwd_Packets",
    "Total_Length_of_Bwd_Packets",
    "Min_Packet_Length",
    "Max_Packet_Length",
    "Packet_Length_Mean",
    "Packet_Length_Std",
    "Flow_IAT_Mean",
    "SYN_Flag_Count",
    "ACK_Flag_Count",
    "RST_Flag_Count"
]

X = df[features]
y = df["Label"]
#Do NOT throw all 80 features “because accuracy”.

In [11]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)


In [12]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [13]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

model.fit(X_train, y_train)


RandomForestClassifier(class_weight='balanced', max_depth=20, n_estimators=200,
                       n_jobs=-1, random_state=42)

Why this works well for IDS:

Handles non-linearity

Robust to noise

Resistant to feature scaling mistakes

Interpretable

In [15]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred = model.predict(X_test)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, digits=4))


[[249487    100]
 [   798  27965]]
              precision    recall  f1-score   support

           0     0.9968    0.9996    0.9982    249587
           1     0.9964    0.9723    0.9842     28763

    accuracy                         0.9968    278350
   macro avg     0.9966    0.9859    0.9912    278350
weighted avg     0.9968    0.9968    0.9968    278350



In [16]:
y_proba = model.predict_proba(X_test)[:, 1]

threshold = 0.8
y_pred_thresh = (y_proba > threshold).astype(int)


In [17]:
import joblib

joblib.dump(model, "ids_model.pkl")
joblib.dump(scaler, "scaler.pkl")
joblib.dump(features, "features.pkl")


['features.pkl']

In [18]:
importances = model.feature_importances_
feature_importance_df = pd.DataFrame({
    "feature": features,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance_df

,feature,importance
1,Total_Fwd_Packets,1.713786e-01
6,Max_Packet_Length,1.428089e-01
3,Total_Length_of_Fwd_Packets,1.422302e-01
7,Packet_Length_Mean,1.420085e-01
8,Packet_Length_Std,1.264933e-01
2,Total_Backward_Packets,7.580964e-02
4,Total_Length_of_Bwd_Packets,5.687859e-02
9,Flow_IAT_Mean,5.169536e-02
0,Flow_Duration,4.218963e-02
5,Min_Packet_Length,3.079657e-02


In [19]:
feature_importance_df[feature_importance_df["importance"] < 0.01]


,feature,importance
11,ACK_Flag_Count,9.501562e-03
10,SYN_Flag_Count,8.209255e-03
12,RST_Flag_Count,7.791384e-10


In [20]:
#Get prediction probabilities
y_proba = model.predict_proba(X_test)[:, 1]

In [21]:
#evaluate multiple thresholds

In [22]:
from sklearn.metrics import confusion_matrix

thresholds = [0.3, 0.5, 0.7, 0.8, 0.9]

results = []

for t in thresholds:
    y_pred_t = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()
    
    recall = tp / (tp + fn)
    fpr = fp / (fp + tn)
    
    results.append((t, recall, fpr))

pd.DataFrame(results, columns=["Threshold", "Attack Recall", "False Positive Rate"])


,Threshold,Attack Recall,False Positive Rate
0,0.3,0.985989,0.017753
1,0.5,0.972256,0.000401
2,0.7,0.971665,0.000333
3,0.8,0.970831,0.000300
4,0.9,0.968606,0.000100


In [23]:
DETECTION_THRESHOLD = 0.8


In [24]:
from sklearn.metrics import classification_report

y_pred_final = (y_proba >= DETECTION_THRESHOLD).astype(int)
print(confusion_matrix(y_test, y_pred_final))
print(classification_report(y_test, y_pred_final, digits=4))


[[249512     75]
 [   839  27924]]
              precision    recall  f1-score   support

           0     0.9966    0.9997    0.9982    249587
           1     0.9973    0.9708    0.9839     28763

    accuracy                         0.9967    278350
   macro avg     0.9970    0.9853    0.9910    278350
weighted avg     0.9967    0.9967    0.9967    278350

